## Model Building and Evaluation
In this section I define the features and target variable, encode categorical columns, split the data into training and testing sets, train a Random Forest Classifier, and evaluate its performance.

**Target Variable:** satisfaction (binary — 1 = Satisfied, 0 = Unsatisfied)

**Independent Variables:** delivery_days, delivered_late, price, freight_value, payment_value, payment_installments, payment_type

In [5]:
import sys
sys.path.append("..")
from utils import *
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Load and clean the data using shared utility functions
df = load_data()
df = clean_data(df) 
print("Data loaded and cleaned:", df.shape)

Data loaded and cleaned: (114842, 22)


### Why Binary Classification
During initial analysis the review score distribution revealed a significant class imbalance with score 5 dominating the dataset. To build a more balanced and business relevant model, the target variable is converted to binary:
- 1 = Satisfied (score 4 or 5)
- 0 = Unsatisfied (score 1, 2, or 3)

In [6]:
# Convert review score to binary classification
# 1 = Satisfied (score 4 or 5), 0 = Unsatisfied (score 1, 2, or 3)
df["satisfaction"] = df["review_score"].apply(
    lambda x: 1 if x >= 4 else 0
)

# Define features and target variable
features = [
    "delivery_days", "delivered_late", "price", "freight_value", "payment_value", "payment_installments", "payment_type"
]

X = df[features]
y = df["satisfaction"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (114842, 7)
Target shape: (114842,)


### Encode Categorical Columns
The payment_type column contains string labels that the model cannot read directly. One-hot encoding converts each unique payment type into a separate binary column so the model can treat each category independently without assuming any order between them.

In [7]:
# Convert payment_type from string to numeric using one hot encoding
X = pd.get_dummies(X, columns=["payment_type"], drop_first=True)

print("Features after encoding:", X.shape)
print("Columns:", X.columns.tolist())

Features after encoding: (114842, 9)
Columns: ['delivery_days', 'delivered_late', 'price', 'freight_value', 'payment_value', 'payment_installments', 'payment_type_credit_card', 'payment_type_debit_card', 'payment_type_voucher']


from the output, we can see that the payment_type column was succesfully converted into three binary columns which arethe credit card,  debit card and voucher. also, Boleto was dropped as the reference categorry. The feature set now contains 9 columns, all which are in numeric format and ready for the model

## Split Data Into Training and Testing Sets
The data is split into two sets which are 80% for training and 20% for testing.

A three way split (training, validation, and test) was not necessary here because we are building a single Random Forest model without extensive hyperparameter tuning across multiple models. The test set is sufficient to evaluate final model performance on unseen data.

In [8]:
# Set target variable
y = df["satisfaction"]

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set size:", X_train.shape)
print("Testing set size:", X_test.shape)

Training set size: (91873, 9)
Testing set size: (22969, 9)


## Train the Model
A Random Forest Classifier is trained on the training set. Random Forest was selected because it handles mixed data types well, does not require feature normalization, and naturally provides feature importance scores that will directly answer our business question in the final notebook.

In [9]:
# Train the Random Forest Classifier
model = RandomForestClassifier(
    n_estimators=100,  # number of trees in the forest
    max_depth=10,      # maximum depth of each tree
    random_state=42    # ensures reproducibility
)

model.fit(X_train, y_train)
print("Model trained successfully")

Model trained successfully


## Evaluate the Model
The trained model is evaluated on the test set to measure how well it performs on data it has never seen before.

In [10]:
# Make predictions on the test set
y_pred = model.predict(X_test)

# Calculate accuracy
print(f"Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

# Detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Model Accuracy: 80.20%

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.22      0.34      5380
           1       0.80      0.98      0.88     17589

    accuracy                           0.80     22969
   macro avg       0.79      0.60      0.61     22969
weighted avg       0.80      0.80      0.76     22969



### Model Performance
The binary classification model achieved 80% accuracy on the test set. The model performs very well at identifying satisfied customers with a recall of 0.98, meaning it correctly flags 98% of satisfied customers. However it struggles to identify unsatisfied customers with a recall of only 0.22, which is expected given that satisfied customers still make up the majority of the dataset even after the binary conversion. Overall the model provides a solid baseline for predicting 
customer satisfaction and will be further analyzed through the confusion matrix and feature importance in the next notebooks.